In [ ]:
import pandas as pd

df = pd.read_csv("mtsamples.csv")

df.head()

,Unnamed: 0,description,medical_specialty,sample_name,transcription,keywords
0,0,A 23-year-old white female presents with comp...,Allergy / Immunology,Allergic Rhinitis,"SUBJECTIVE:, This 23-year-old white female pr...","allergy / immunology, allergic rhinitis, aller..."
1,1,Consult for laparoscopic gastric bypass.,Bariatrics,Laparoscopic Gastric Bypass Consult - 2,"PAST MEDICAL HISTORY:, He has difficulty climb...","bariatrics, laparoscopic gastric bypass, weigh..."
2,2,Consult for laparoscopic gastric bypass.,Bariatrics,Laparoscopic Gastric Bypass Consult - 1,"HISTORY OF PRESENT ILLNESS: , I have seen ABC ...","bariatrics, laparoscopic gastric bypass, heart..."
3,3,2-D M-Mode. Doppler.,Cardiovascular / Pulmonary,2-D Echocardiogram - 1,"2-D M-MODE: , ,1. Left atrial enlargement wit...","cardiovascular / pulmonary, 2-d m-mode, dopple..."
4,4,2-D Echocardiogram,Cardiovascular / Pulmonary,2-D Echocardiogram - 2,1. The left ventricular cavity size and wall ...,"cardiovascular / pulmonary, 2-d, doppler, echo..."


In [ ]:
df = df[['transcription', 'medical_specialty']]

In [ ]:
df = df.dropna(subset=['transcription', 'medical_specialty'])

df['transcription'] = (
    df['transcription']
    .astype(str)
    .str.replace('\n', ' ')
    .str.replace('\r', ' ')
    .str.replace(r'\s+', ' ', regex=True)
    .str.strip()
)

df['medical_specialty'] = df['medical_specialty'].astype(str).str.strip()

df = df.drop_duplicates(subset=['transcription'])

In [ ]:
counts = df['medical_specialty'].value_counts()

valid_departments = counts[counts >= 50].index

df = df[df['medical_specialty'].isin(valid_departments)]

In [ ]:
label2id = {label: i for i, label in enumerate(sorted(df['medical_specialty'].unique()))}
id2label = {i: label for label, i in label2id.items()}

df['label_id'] = df['medical_specialty'].map(label2id)

In [ ]:
from sklearn.model_selection import train_test_split

train_df, test_df = train_test_split(
    df,
    test_size=0.2,
)

print("Train:", train_df.shape)
print("Test:", test_df.shape)

Train: (1569, 3)
Test: (393, 3)


In [ ]:
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.linear_model import LogisticRegression
from sklearn.pipeline import Pipeline

lr_model = Pipeline([
    ("tfidf", TfidfVectorizer(
        max_features=5000,
        stop_words="english",
         ngram_range=(1, 2)
    )),
    ("classifier", LogisticRegression(
        max_iter=1000,
          C=1.0,
        class_weight="balanced"
    ))
])

In [ ]:
lr_model.fit(
    train_df["transcription"],
    train_df["label_id"]
)

Pipeline(steps=[('tfidf',
                 TfidfVectorizer(max_features=5000, ngram_range=(1, 2),
                                 stop_words='english')),
                ('classifier',
                 LogisticRegression(class_weight='balanced', max_iter=1000))])

In [ ]:
from sklearn.metrics import accuracy_score, precision_recall_fscore_support, classification_report

y_pred_lr = lr_model.predict(test_df["transcription"])
y_true = test_df["label_id"]

accuracy_lr = accuracy_score(y_true, y_pred_lr)

precision_lr, recall_lr, f1_lr, _ = precision_recall_fscore_support(
    y_true,
    y_pred_lr,
    average="weighted",
    zero_division=0
)

print("Accuracy:", accuracy_lr)
print("Precision:", precision_lr)
print("Recall:", recall_lr)
print("F1-score:", f1_lr)

Accuracy: 0.8880407124681934
Precision: 0.892154323794229
Recall: 0.8880407124681934
F1-score: 0.889258620230519


In [ ]:
print(classification_report(
    y_true,
    y_pred_lr,
    target_names=[id2label[i] for i in range(len(id2label))],
    zero_division=0
))

                               precision    recall  f1-score   support

   Consult - History and Phy.       0.69      0.69      0.69        13
             General Medicine       0.73      0.67      0.70        24
                    Neurology       0.50      0.50      0.50         8
                   Orthopedic       0.82      0.82      0.82        11
        Pediatrics - Neonatal       0.50      0.50      0.50         8
      Psychiatry / Psychology       0.93      1.00      0.97        14
                    Radiology       0.84      0.91      0.88        54
SOAP / Chart / Progress Notes       0.70      0.82      0.75        28
                      Surgery       0.99      0.96      0.98       206
                      Urology       0.89      0.89      0.89        27

                     accuracy                           0.89       393
                    macro avg       0.76      0.78      0.77       393
                 weighted avg       0.89      0.89      0.89       393



In [ ]:
from sklearn.ensemble import RandomForestClassifier

rf_model = Pipeline([
    ("tfidf", TfidfVectorizer(
        max_features=5000,
        stop_words="english",
        ngram_range=(1, 2)
    )),
    ("classifier", RandomForestClassifier(
        n_estimators=100,
        class_weight="balanced"
    ))
])

In [ ]:
rf_model.fit(
    train_df["transcription"],
    train_df["label_id"]
)

Pipeline(steps=[('tfidf',
                 TfidfVectorizer(max_features=5000, ngram_range=(1, 2),
                                 stop_words='english')),
                ('classifier',
                 RandomForestClassifier(class_weight='balanced'))])

In [ ]:
y_pred_rf = rf_model.predict(test_df["transcription"])
y_true = test_df["label_id"]

accuracy_rf = accuracy_score(y_true, y_pred_rf)

precision_rf, recall_rf, f1_rf, _ = precision_recall_fscore_support(
    y_true,
    y_pred_rf,
    average="weighted",
    zero_division=0
)

print("Accuracy:", accuracy_rf)
print("Precision:", precision_rf)
print("Recall:", recall_rf)
print("F1-score:", f1_rf)

Accuracy: 0.821882951653944
Precision: 0.8451239337315286
Recall: 0.821882951653944
F1-score: 0.7959115300012531


In [ ]:
print(classification_report(
    y_true,
    y_pred_rf,
    target_names=[id2label[i] for i in range(len(id2label))],
    zero_division=0
))

                               precision    recall  f1-score   support

   Consult - History and Phy.       0.67      0.15      0.25        13
             General Medicine       0.45      0.79      0.58        24
                    Neurology       0.67      0.25      0.36         8
                   Orthopedic       1.00      0.09      0.17        11
        Pediatrics - Neonatal       1.00      0.38      0.55         8
      Psychiatry / Psychology       1.00      0.86      0.92        14
                    Radiology       0.91      0.93      0.92        54
SOAP / Chart / Progress Notes       0.71      0.71      0.71        28
                      Surgery       0.86      1.00      0.93       206
                      Urology       1.00      0.33      0.50        27

                     accuracy                           0.82       393
                    macro avg       0.83      0.55      0.59       393
                 weighted avg       0.85      0.82      0.80       393



In [ ]:
results = pd.DataFrame({
    "Model": ["Logistic Regression", "Random Forest"],
    "Accuracy": [accuracy_lr, accuracy_rf],
    "Precision": [precision_lr, precision_rf],
    "Recall": [recall_lr, recall_rf],
    "F1-Score": [f1_lr, f1_rf]
})

print(results)

                 Model  Accuracy  Precision    Recall  F1-Score
0  Logistic Regression  0.888041   0.892154  0.888041  0.889259
1        Random Forest  0.821883   0.845124  0.821883  0.795912


In [ ]:
import joblib

joblib.dump(lr_model, "winning_model.joblib")
joblib.dump(id2label, "id2label.joblib")

['id2label.joblib']